# Z2005 — Week 2: Stacks and Queues
A self-study notebook covering the Stack (LIFO) and Queue (FIFO) abstract data types, array- and deque-backed implementations, the naive queue's hidden O(n) trap, the circular buffer, and stack/queue applications.

## Learning Objectives

By the end of this notebook you will be able to:

- Explain LIFO and FIFO and implement a `Stack` and a `Queue`, each with the correct O(1) operations.
- Identify exactly where a naive array-backed queue hides an O(n) cost, and fix it with `collections.deque`.
- Implement a fixed-capacity `CircularQueue` using modular arithmetic, resolving the full-vs-empty ambiguity.
- Use a stack to check balanced parentheses and to evaluate postfix expressions.
- Convert infix expressions to postfix with the shunting-yard algorithm.

## How to use this notebook

Run the cells top to bottom. `# TODO` cells are for you; `assert`-based cells are self-checks. Solutions are at the end.

## 1. `Stack`: Last In, First Out

A cafeteria plate dispenser: plates go on top, plates come off the top, and you can never pull one from the middle without disturbing everything above it. That single constraint -- touch only the top -- is the entire definition of a stack. **LIFO**: the last item pushed is the first one popped.

A stack built on Week 1's dynamic array must call `list.pop()` (default: removes from the end) rather than `list.pop(0)` (removes from the start). Week 1 showed removing from the end is O(1); removing from the start requires shifting every remaining element, O(n). Using the wrong end silently turns every stack operation into an O(n) operation.

| Operation | Cost | Why |
|---|---|---|
| `push(x)` | O(1) amortized | doubling dynamic-array append |
| `pop()` | O(1) | removing from the end |
| `peek()` | O(1) | array index access |
| `is_empty()` | O(1) | reading `len()`, not scanning |

You have already been using a real stack since Week 0: the **call stack**. Calling `a()` pushes a's frame (where to resume, local variables); `a()` calling `b()` pushes b's frame on top of a's; `c()` returns first, popping its frame -- LIFO, exactly.

In [ ]:
class Stack:
    def __init__(self):
        self._items = []   # Week 1's dynamic array, doubling under the hood

    def push(self, x):
        self._items.append(x)   # amortized O(1) -- Week 1

    def pop(self):
        if not self._items:
            raise IndexError("pop from an empty stack")
        return self._items.pop()   # removes and returns the LAST element -- O(1)

    def peek(self):
        if not self._items:
            raise IndexError("peek at an empty stack")
        return self._items[-1]

    def is_empty(self):
        return len(self._items) == 0


s = Stack()
s.push(1); s.push(2); s.push(3)
assert s.pop() == 3
assert s.peek() == 2
assert not s.is_empty()
print("Stack checks passed")

### Application: checking balanced parentheses

Is `"(a + [b * (c - d)]) / e"` correctly balanced? The rule: every closing bracket must match the *most recently opened, still-unclosed* bracket -- exactly what a stack's top holds. Push every opening bracket; on a closing bracket, pop and check it matches. If the stack is empty at a close, or the popped bracket doesn't match, the string is unbalanced. At the end, the stack must be empty (no unclosed brackets left over).

**Why not just count brackets?** Counting `(` versus `)` separately catches count mismatches like `"(()"`, but not *order* mismatches: `"([)]"` has equal counts of every bracket type, so a counting approach would wrongly report it balanced. The stack encodes order, not just quantity -- that is the structural reason it is the correct tool here.

In [ ]:
def is_balanced(s):
    pairs = {')': '(', ']': '[', '}': '{'}
    stack = Stack()
    for ch in s:
        if ch in "([{":
            stack.push(ch)
        elif ch in ")]}":
            # empty stack at a close, OR a mismatched pair -> unbalanced
            if stack.is_empty() or stack.pop() != pairs[ch]:
                return False
    return stack.is_empty()   # anything left un-closed also means unbalanced


assert is_balanced("([{}])") == True
assert is_balanced("([)]") == False        # order mismatch, same bracket counts
assert is_balanced("(a + [b * (c - d)]) / e") == True
print("Balanced-parentheses checks passed")

## 2. `Queue`: First In, First Out, and the naive queue's hidden trap

A cafeteria line: people join at the back, are served and leave from the front, and the first person to join is the first person served. **FIFO**: `enqueue` adds at the back, `dequeue` removes from the front -- the mirror image of a stack.

A naive queue backed by a plain list looks reasonable: `enqueue` uses `append` (O(1) amortized, the cheap end). But `dequeue` using `list.pop(0)` removes from index 0 -- Week 1's *worst case* for deletion. Every remaining element must shift left by one to close the gap at the front, so each `dequeue()` call costs O(n), not O(1). Processing `n` items one at a time this way costs O(n²) total -- the exact same shape of mistake as "grow by 1" from Week 1, in a different operation.

In [ ]:
class NaiveQueue:
    def __init__(self):
        self._items = []

    def enqueue(self, x):
        self._items.append(x)      # add to the back -- O(1) amortized

    def dequeue(self):
        if not self._items:
            raise IndexError("dequeue from an empty queue")
        return self._items.pop(0)  # remove from the FRONT -- O(n), shifts everything left


import timeit

def time_naive_dequeue(n):
    q = list(range(n))
    def run():
        for _ in range(n):
            q.pop(0)
    return timeit.timeit(run, number=1)

for n in (2_000, 4_000, 8_000):
    t = time_naive_dequeue(n)
    print(f"n={n:>6}  naive dequeue total time={t:.4f}s")

The timing should grow noticeably faster than linearly as `n` doubles -- evidence of the O(n²) total cost predicted above for the naive, `pop(0)`-based queue.

**The fix:** `collections.deque` is a doubly linked list of fixed-size blocks, not one contiguous array. `popleft()` needs no shifting at all -- the "front" is simply relabeled, not physically moved. Both `enqueue` and `dequeue` become genuinely O(1).

In [ ]:
from collections import deque

class Queue:
    def __init__(self):
        self._items = deque()

    def enqueue(self, x):
        self._items.append(x)      # add to the back -- O(1)

    def dequeue(self):
        if not self._items:
            raise IndexError("dequeue from an empty queue")
        return self._items.popleft()   # remove from the front -- genuinely O(1)

    def is_empty(self):
        return len(self._items) == 0


jobs = Queue()
jobs.enqueue("report.pdf")
jobs.enqueue("photo.png")
jobs.enqueue("invoice.pdf")
assert jobs.dequeue() == "report.pdf"
assert jobs.dequeue() == "photo.png"
assert jobs.dequeue() == "invoice.pdf"
print("deque-backed Queue checks passed, FIFO order confirmed")

## 3. `CircularQueue`: a queue backed by one fixed array

`deque` is an excellent general-purpose queue, but it allocates memory dynamically as needed -- no hard upper bound. Some systems (embedded devices, audio buffers, network packet buffers) need a **fixed**, pre-allocated memory budget known in advance. A **circular buffer** gives exactly that: a fixed-size array where `front` and `back` markers *wrap around* using modular arithmetic (`% capacity`) instead of shifting elements. No element ever physically moves -- only the markers move -- which is why both operations stay O(1).

**The full-vs-empty ambiguity:** an empty buffer has `front == back` (0 elements); a *completely full* buffer, after wrapping, can also end up with `front == back` (capacity elements). Comparing the markers alone cannot distinguish the two. The fix: track `count` explicitly as a tie-breaker.

In [ ]:
class CircularQueue:
    def __init__(self, capacity):
        self.capacity = capacity
        self._data = [None] * capacity
        self.front = 0
        self.back = 0
        self.count = 0   # the tie-breaker: resolves the front==back ambiguity

    def is_empty(self):
        return self.count == 0

    def is_full(self):
        return self.count == self.capacity

    def enqueue(self, x):
        if self.is_full():
            raise OverflowError("circular queue is full")
        self._data[self.back] = x
        self.back = (self.back + 1) % self.capacity   # wrap around, don't shift
        self.count += 1

    def dequeue(self):
        if self.is_empty():
            raise IndexError("dequeue from an empty queue")
        x = self._data[self.front]
        self.front = (self.front + 1) % self.capacity
        self.count -= 1
        return x


cq = CircularQueue(3)
cq.enqueue('A'); cq.enqueue('B'); cq.enqueue('C')
assert cq.is_full()
assert cq.dequeue() == 'A'
cq.enqueue('D')   # wraps into slot 0, previously held 'A'
assert cq.dequeue() == 'B'
assert cq.dequeue() == 'C'
assert cq.dequeue() == 'D'
assert cq.is_empty()
print("CircularQueue checks passed, including a full wrap-around")

## 4. Expression evaluation: postfix and shunting-yard

`"3 + 4 * 2"` -- is the answer 14 or 11? Correct arithmetic evaluates multiplication first: `3 + (4 * 2) = 11`. A naive left-to-right reader would wrongly compute `(3 + 4) * 2 = 14`. **Postfix** notation (Reverse Polish Notation), `3 4 2 * +`, places every operator *after* its operands and needs no precedence rules at all: push numbers as they arrive; when an operator appears, pop its two most recent operands, compute, and push the result back.

In [ ]:
def evaluate_postfix(tokens):
    stack = []
    for tok in tokens:
        if tok in "+-*/":
            b = stack.pop()   # second operand -- popped FIRST
            a = stack.pop()   # first operand
            if tok == '+': stack.append(a + b)
            elif tok == '-': stack.append(a - b)
            elif tok == '*': stack.append(a * b)
            elif tok == '/': stack.append(a / b)
        else:
            stack.append(float(tok))
    return stack.pop()


assert evaluate_postfix(['3', '4', '2', '*', '+']) == 11.0
assert evaluate_postfix(['10', '2', '3', '-', '*']) == -10.0
print("Postfix evaluation checks passed")

**Shunting-yard** (Dijkstra, named for a railway shunting yard's rearranging of train cars) converts human-written infix to postfix using a stack to temporarily *hold* operators until their correct output position is known: an operator is only sent to the output once every higher- (or equal-) precedence operator already on the stack has been output first.

In [ ]:
PRECEDENCE = {'+': 1, '-': 1, '*': 2, '/': 2}

def infix_to_postfix(tokens):
    output = []
    ops = []
    for tok in tokens:
        if tok not in PRECEDENCE:
            output.append(tok)
        else:
            # pop every operator of >= precedence before pushing this one
            while (ops and ops[-1] in PRECEDENCE
                   and PRECEDENCE[ops[-1]] >= PRECEDENCE[tok]):
                output.append(ops.pop())
            ops.append(tok)
    while ops:
        output.append(ops.pop())
    return output

def calculate(infix_tokens):
    return evaluate_postfix(infix_to_postfix(infix_tokens))


assert infix_to_postfix(['3', '+', '4', '*', '2']) == ['3', '4', '2', '*', '+']
assert calculate(['3', '+', '4', '*', '2']) == 11.0
assert calculate(['10', '-', '2', '*', '3']) == 4.0
print("Shunting-yard and calculator checks passed")

## Exercises

### Exercise 1 — build your own `Stack`

Complete `MyStack`, backed by a plain Python list, with `push`, `pop`, `peek`, and `is_empty`, matching the reference in Section 1 (raise `IndexError` on `pop`/`peek` of an empty stack).

In [ ]:
class MyStack:
    def __init__(self):
        self._items = []

    def push(self, x):
        # TODO: append x to self._items
        raise NotImplementedError

    def pop(self):
        # TODO: raise IndexError if empty, else remove and return the last item
        raise NotImplementedError

    def peek(self):
        # TODO: raise IndexError if empty, else return (without removing) the last item
        raise NotImplementedError

    def is_empty(self):
        # TODO: return True if there are no items
        raise NotImplementedError

**Self-check — Exercise 1**

In [ ]:
st = MyStack()
st.push(10); st.push(20); st.push(30)
assert st.peek() == 30
assert st.pop() == 30
assert st.pop() == 20
assert not st.is_empty()
assert st.pop() == 10
assert st.is_empty()
try:
    st.pop()
    raise AssertionError("pop on an empty stack should raise IndexError")
except IndexError:
    pass
print("\u2705 Exercise 1 passed")

### Exercise 2 — `is_valid_html_tags`, a balanced-parentheses variant

Write `is_valid_html_tags(tags)`, where `tags` is a list of strings like `['<div>', '</div>']` or `['<p>', '<b>', '</b>', '</p>']`. A closing tag `</X>` must match the most recently opened, still-unclosed `<X>`. Return `True` if every tag is properly nested and closed, `False` otherwise. (Reuse the same stack-based shape as `is_balanced`.)

Example: `is_valid_html_tags(['<div>', '<p>', '</p>', '</div>'])` is `True`; `is_valid_html_tags(['<div>', '<p>', '</div>', '</p>'])` is `False` (wrong order).

In [ ]:
def is_valid_html_tags(tags):
    # TODO: push opening tags (e.g. '<div>'); on a closing tag (e.g. '</div>'),
    # pop and check it matches the corresponding opening tag ('<' + closing_name + '>').
    # Return False on any mismatch or premature close; at the end, the stack must be empty.
    raise NotImplementedError

**Self-check — Exercise 2**

In [ ]:
assert is_valid_html_tags(['<div>', '<p>', '</p>', '</div>']) is True
assert is_valid_html_tags(['<div>', '<p>', '</div>', '</p>']) is False
assert is_valid_html_tags(['<a>', '</a>', '<b>', '</b>']) is True
assert is_valid_html_tags(['<a>']) is False   # never closed
print("\u2705 Exercise 2 passed")

### Exercise 3 — `Queue` backed by `deque`

Complete `MyQueue`, backed by `collections.deque`, with `enqueue`, `dequeue`, and `is_empty`, matching Section 2's reference.

In [ ]:
from collections import deque as _deque

class MyQueue:
    def __init__(self):
        self._items = _deque()

    def enqueue(self, x):
        # TODO: add x to the back
        raise NotImplementedError

    def dequeue(self):
        # TODO: raise IndexError if empty, else remove and return the front item
        raise NotImplementedError

    def is_empty(self):
        # TODO: return True if there are no items
        raise NotImplementedError

**Self-check — Exercise 3**

In [ ]:
q = MyQueue()
q.enqueue('a'); q.enqueue('b'); q.enqueue('c')
assert q.dequeue() == 'a'
assert q.dequeue() == 'b'
assert not q.is_empty()
assert q.dequeue() == 'c'
assert q.is_empty()
print("\u2705 Exercise 3 passed")

### Exercise 4 — `CircularQueue.enqueue_overwrite`

Add an alternative enqueue policy to a circular queue: instead of raising `OverflowError` when full, `enqueue_overwrite(x)` should overwrite the *oldest* element (advancing `front` by one, since that slot's old value is now lost) -- the policy real logging ring buffers use for "keep only the last N entries." Complete the subclass below.

In [ ]:
class OverwritingCircularQueue(CircularQueue):
    def enqueue_overwrite(self, x):
        # TODO: if the queue is full, first advance front by one (% capacity) and
        # decrement count by one -- this "forgets" the oldest element. Then perform
        # a normal enqueue(x).
        raise NotImplementedError

**Self-check — Exercise 4**

In [ ]:
ocq = OverwritingCircularQueue(3)
ocq.enqueue('A'); ocq.enqueue('B'); ocq.enqueue('C')
assert ocq.is_full()
ocq.enqueue_overwrite('D')   # 'A' (the oldest) should be lost
assert ocq.count == 3
assert ocq.dequeue() == 'B'
assert ocq.dequeue() == 'C'
assert ocq.dequeue() == 'D'
print("\u2705 Exercise 4 passed")

### Exercise 5 (harder) — extend `evaluate_postfix` with `^` (exponentiation)

Add support for the `^` operator (right-associative exponentiation, e.g. `2 ^ 3 = 8`) to a copy of `evaluate_postfix`.

In [ ]:
def evaluate_postfix_ext(tokens):
    stack = []
    for tok in tokens:
        if tok in "+-*/^":
            b = stack.pop()
            a = stack.pop()
            # TODO: handle '+', '-', '*', '/' as before, and now also '^' (a ** b)
            raise NotImplementedError
        else:
            stack.append(float(tok))
    return stack.pop()

**Self-check — Exercise 5**

In [ ]:
assert evaluate_postfix_ext(['2', '3', '^']) == 8.0
assert evaluate_postfix_ext(['2', '3', '^', '1', '+']) == 9.0
print("\u2705 Exercise 5 passed")

## Quiz

**1. What is the one-sentence difference between a stack and a queue?**
<details><summary>Show answer</summary>A stack is LIFO (last in, first out; add and remove at the same end, the top); a queue is FIFO (first in, first out; add at the back, remove from the front).</details>

**2. Why does the naive array-backed queue's `dequeue` cost O(n), and how does `collections.deque` fix it?</summary>**
<details><summary>Show answer</summary><code>list.pop(0)</code> removes the first element, which forces every remaining element to shift left by one to close the gap -- O(n). <code>deque</code> is a doubly linked list of blocks, so <code>popleft()</code> just relabels the front; nothing physically moves, giving genuine O(1).</details>

**3. In a circular buffer, why can't you tell empty and full apart just by checking `front == back`?**
<details><summary>Show answer</summary>Both an empty buffer and a completely full buffer (after wrapping) can produce <code>front == back</code> -- the markers alone are ambiguous. An explicit <code>count</code> field resolves it.</details>

**4. Why does counting brackets (number of `(` vs `)`) fail to detect `"([)]"` as unbalanced?**
<details><summary>Show answer</summary>Counting only tracks quantity, and <code>"([)]"</code> has equal counts of every bracket type. It says nothing about ORDER, so it misses that the <code>)</code> closes before the matching <code>[</code> is closed. A stack encodes order because it always exposes the most-recently-opened, still-unclosed bracket at the top.</details>

## Solutions (try the exercises yourself first!)

In [ ]:
# --- Exercise 1 solution ---
class MyStack:
    def __init__(self):
        self._items = []

    def push(self, x):
        self._items.append(x)

    def pop(self):
        if not self._items:
            raise IndexError("pop from an empty stack")
        return self._items.pop()

    def peek(self):
        if not self._items:
            raise IndexError("peek at an empty stack")
        return self._items[-1]

    def is_empty(self):
        return len(self._items) == 0


st = MyStack()
st.push(1); st.push(2)
assert st.pop() == 2
print("Exercise 1 solution verified")

In [ ]:
# --- Exercise 2 solution ---
def is_valid_html_tags(tags):
    stack = []
    for tag in tags:
        if not tag.startswith('</'):
            stack.append(tag)
        else:
            name = tag[2:-1]
            expected = f"<{name}>"
            if not stack or stack.pop() != expected:
                return False
    return len(stack) == 0


assert is_valid_html_tags(['<div>', '<p>', '</p>', '</div>']) is True
assert is_valid_html_tags(['<div>', '<p>', '</div>', '</p>']) is False
print("Exercise 2 solution verified")

In [ ]:
# --- Exercise 3 solution ---
class MyQueue:
    def __init__(self):
        self._items = _deque()

    def enqueue(self, x):
        self._items.append(x)

    def dequeue(self):
        if not self._items:
            raise IndexError("dequeue from an empty queue")
        return self._items.popleft()

    def is_empty(self):
        return len(self._items) == 0


q = MyQueue()
q.enqueue(1); q.enqueue(2)
assert q.dequeue() == 1
print("Exercise 3 solution verified")

In [ ]:
# --- Exercise 4 solution ---
class OverwritingCircularQueue(CircularQueue):
    def enqueue_overwrite(self, x):
        if self.is_full():
            self.front = (self.front + 1) % self.capacity   # forget the oldest element
            self.count -= 1
        self.enqueue(x)


ocq = OverwritingCircularQueue(3)
ocq.enqueue('A'); ocq.enqueue('B'); ocq.enqueue('C')
ocq.enqueue_overwrite('D')
assert ocq.dequeue() == 'B'
print("Exercise 4 solution verified")

In [ ]:
# --- Exercise 5 solution ---
def evaluate_postfix_ext(tokens):
    stack = []
    for tok in tokens:
        if tok in "+-*/^":
            b = stack.pop()
            a = stack.pop()
            if tok == '+': stack.append(a + b)
            elif tok == '-': stack.append(a - b)
            elif tok == '*': stack.append(a * b)
            elif tok == '/': stack.append(a / b)
            elif tok == '^': stack.append(a ** b)
        else:
            stack.append(float(tok))
    return stack.pop()


assert evaluate_postfix_ext(['2', '3', '^']) == 8.0
print("Exercise 5 solution verified")

## MTech Extension — the monotonic stack, and `deque` vs. list-based queue complexity

**The monotonic stack pattern.** Problem: for each element in an array, find the *next greater element* to its right (or -1 if none exists). A naive approach scans rightward from each position -- O(n²) overall. A **monotonic stack** (kept in strictly decreasing order of value) solves it in O(n): push each index; before pushing, pop off every index whose value is *smaller* than the current element (each of those indices has just found its answer -- the current element is their next greater element). Each index is pushed once and popped at most once across the whole run, so total work is O(n), not O(n²) -- the same amortized-cost reasoning as Week 1's dynamic array, applied to a different structure. This pattern recurs in stock-span problems, histogram-area problems, and temperature-tracking problems: same shape, different surface.

**Complexity of `deque` vs. list-based queue, formalized.** The table below (from Section 2/lecture) is worth restating precisely for a systems-level audience: `deque` is a doubly linked list of fixed-size *blocks* (not single nodes, and not one contiguous array). This block structure is why it gets O(1) at both ends without the memory-fragmentation cost of a pure node-per-element linked list, while still paying O(n) for random access (must walk block-by-block from whichever end is closer). A plain `list` is the mirror image: O(1) random access via the address-arithmetic formula from Week 1, but O(n) for front insertion/removal because of shifting.

| Operation | `list` | `deque` |
|---|---|---|
| Append at end | O(1) amortized | O(1) |
| Append at start | O(n) | O(1) |
| Access by index | O(1) | O(n) |
| Contiguous in memory | yes | no (blocks) |

The cell below implements and times the monotonic-stack solution against the naive O(n²) approach.

In [ ]:
def next_greater_naive(arr):
    # O(n^2): for each element, scan rightward until a greater one is found
    n = len(arr)
    result = [-1] * n
    for i in range(n):
        for j in range(i + 1, n):
            if arr[j] > arr[i]:
                result[i] = arr[j]
                break
    return result

def next_greater_monotonic(arr):
    # O(n): each index is pushed once and popped at most once
    result = [-1] * len(arr)
    stack = []   # holds indices, kept with DECREASING values
    for i, x in enumerate(arr):
        while stack and arr[stack[-1]] < x:
            result[stack.pop()] = x   # current x is the next-greater for this popped index
        stack.append(i)
    return result


sample = [2, 1, 2, 4, 3]
assert next_greater_naive(sample) == [4, 2, 4, -1, -1]
assert next_greater_monotonic(sample) == [4, 2, 4, -1, -1]
print("Both approaches agree:", next_greater_monotonic(sample))

import timeit, random
random.seed(0)
for n in (500, 1000, 2000):
    data = [random.randint(0, n) for _ in range(n)]
    t_naive = timeit.timeit(lambda: next_greater_naive(data), number=3)
    t_mono = timeit.timeit(lambda: next_greater_monotonic(data), number=3)
    print(f"n={n:>5}  naive O(n^2)={t_naive:.4f}s   monotonic O(n)={t_mono:.4f}s")

As `n` doubles, the naive approach's time should grow noticeably faster than the monotonic stack's -- direct evidence of O(n²) versus O(n).